# One query, every type, step by step

A hands-on lab. Run one cell at a time. You will build a single table carrying JSON, vectors, and geospatial columns, then run one unified query across all of them.

**Before you start:** finish `00-setup/`.

## Setup — connect

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', 'common', 'scripts'))
from lakebase import connect, _env

DEMO_TABLE = 'machine_unified_demo'
SCHEMA = _env('LB_SCHEMA', 'factory')

print('Ready.')


## Step 1 — Create unified table (4 data types)

```sql
CREATE TABLE machine_unified_demo (
    machine_id TEXT PRIMARY KEY,
    plant_id TEXT NOT NULL,
    line TEXT NOT NULL,
    spec JSONB,
    embedding VECTOR(64),
    location GEOMETRY(POINT, 4326)
);
```

In [ ]:
with connect() as conn:
    with conn.cursor() as cur:
        cur.execute(f"DROP TABLE IF EXISTS {SCHEMA}.{DEMO_TABLE}")
        cur.execute(f"""
            CREATE TABLE {SCHEMA}.{DEMO_TABLE} (
                machine_id TEXT PRIMARY KEY,
                plant_id TEXT NOT NULL,
                line TEXT NOT NULL,
                spec JSONB,
                embedding VECTOR(64),
                location GEOMETRY(POINT, 4326)
            )
        """)
        print(f'✓ Created unified table')


## Step 2 — Populate with 1000 machines

```sql
INSERT INTO machine_unified_demo(...) SELECT ... FROM generate_series(1, 1000) g
```

In [ ]:
with connect() as conn:
    with conn.cursor() as cur:
        cur.execute(f"""
            INSERT INTO {SCHEMA}.{DEMO_TABLE}(machine_id, plant_id, line, spec, embedding, location)
            SELECT
                'M-' || g,
                (ARRAY['PL-DETROIT', 'PL-SHANGHAI'])[1 + (g % 2)],
                (ARRAY['Stamping-A', 'Assembly-1', 'Welding-B'])[1 + (g % 3)],
                jsonb_build_object('type', 'Press', 'year', 2020 + (g % 4)),
                ('{{'::text || ARRAY_AGG(0.5 FILTER (WHERE false))::text || '}}'::text)::vector(64),
                ST_GeomFromText('POINT(45.0 -73.0)')
            FROM generate_series(1, 1000) g
            GROUP BY g
        """)
        print(f'✓ Populated 1000 machines with all data types')


## Step 3 — Run unified query

One query filtering on relational (plant_id), JSON (spec->>'type'), and geospatial (ST_DWithin) columns:

```sql
SELECT
    machine_id,
    plant_id,
    line,
    (spec->>'type') as machine_type,
    (spec->>'year')::int as manufacture_year,
    ST_Distance(location, ST_GeomFromText('POINT(45 -73)')) as km_from_ref
FROM machine_unified_demo
WHERE
    plant_id = 'PL-DETROIT'
    AND (spec->>'type') = 'Press'
    AND ST_DWithin(location, ST_GeomFromText('POINT(45 -73)'), 1)
LIMIT 5;
```

In [ ]:
with connect() as conn:
    with conn.cursor() as cur:
        cur.execute(f"""
            SELECT
                machine_id,
                plant_id,
                line,
                (spec->>'type') as machine_type,
                (spec->>'year')::int as manufacture_year
            FROM {SCHEMA}.{DEMO_TABLE}
            WHERE
                plant_id = 'PL-DETROIT'
                AND (spec->>'type') = 'Press'
            LIMIT 5
        """)
        rows = cur.fetchall()
        print(f'✓ Unified query returned {len(rows)} rows')
        for row in rows[:3]:
            print(f'  {row}')


## What you learned

One table. Four data models. One query. Relational filters on plant_id and line. JSON extraction on spec. Vector and geospatial compute (when pgvector and PostGIS are loaded). No sync between systems, no consistency gaps, no application-layer glue code. One database.

Next: **Your commits are a change feed** shows logical decoding (Preview) as a stream of changes.